# 02 - Implementación de XGBoost, LightGBM, CatBoost y Benchmark Comparativo

## Curso de Machine Learning: Modelos de Clasificación

En este cuaderno profundizaremos en la configuración práctica, hiperparámetros de regularización, parada temprana (*Early Stopping*) y manejo nativo de valores faltantes de la familia moderna de **Gradient Boosting**: **XGBoost**, **LightGBM** (y su equivalente nativo en Scikit-Learn `HistGradientBoostingClassifier`) y **CatBoost**. Finalizaremos con un **benchmark integral** frente al resto de clasificadores del curso.

---

### Contenidos
1. Hiperparámetros Críticos de Regularización en Boosting Moderno
2. Configuración de **XGBoost**, **LightGBM / HistGradientBoosting** y **CatBoost**
3. Parada Temprana (*Early Stopping*) y Prevención del Sobreajuste
4. Robustez Nativa ante Valores Faltantes (`NaN`) sin Imputación Previa
5. Fronteras de Decisión en 2D e Importancia de Variables (Gain vs Permutation)
6. Benchmark Integral de Clasificación: Precisión, ROC-AUC y Tiempos de Cómputo


In [ ]:
import time
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer, make_moons
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier, HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.inspection import DecisionBoundaryDisplay, permutation_importance
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, classification_report

# Verificacion dinamica de librerias externas (XGBoost, LightGBM, CatBoost)
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False

try:
    from lightgbm import LGBMClassifier
    HAS_LGBM = True
except ImportError:
    HAS_LGBM = False

try:
    from catboost import CatBoostClassifier
    HAS_CAT = True
except ImportError:
    HAS_CAT = False

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
np.random.seed(42)
print(f"Estado de librerias -> XGBoost: {HAS_XGB} | LightGBM: {HAS_LGBM} | CatBoost: {HAS_CAT}")
print("Nota: HistGradientBoostingClassifier de Scikit-Learn implementa de forma nativa la arquitectura de LightGBM.")


## 1. Hiperparámetros de Regularización en la Función Objetivo

En **XGBoost** y **LightGBM**, la función objetivo en el paso $m$ combina la pérdida de clasificación con un término de penalización $\Omega(h_m)$ sobre la complejidad del árbol:

$$\mathcal{Obj}^{(m)} = \sum_{i=1}^N \left[ g_i h_m(\mathbf{x}_i) + \frac{1}{2} H_i h_m^2(\mathbf{x}_i) \right] + \gamma T + \frac{1}{2}\lambda \sum_{j=1}^T w_j^2 + \alpha \sum_{j=1}^T |w_j|$$

Donde:
- $g_i = \partial_{\hat{y}} \mathcal{L}(y_i, \hat{y}_i)$ es el **Gradiente de primer orden** ($\hat{p}_i - y_i$).
- $H_i = \partial^2_{\hat{y}} \mathcal{L}(y_i, \hat{y}_i)$ es el **Hessiano (derivada de segundo orden)**: $\hat{p}_i(1 - \hat{p}_i)$.
- $T$: Número de hojas del árbol, penalizado por `gamma` ($\gamma$, ganancia mínima para dividir un nodo).
- `reg_lambda` ($\lambda$ / `l2_regularization`): Regularización $L_2$ que suaviza los pesos $w_j$ de las hojas.
- `reg_alpha` ($\alpha$): Regularización $L_1$ que fomenta dispersión en los pesos de las hojas.
- `colsample_bytree`: Muestreo aleatorio de columnas por árbol (similar a Random Forest).


In [ ]:
# Instanciacion modular de la familia Gradient Boosting (con fallback automatico si alguna libreria no esta instalada)
cancer = load_breast_cancer()
X, y = cancer.data, cancer.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

modelos_gb = {
    'Sklearn GradientBoosting': GradientBoostingClassifier(
        n_estimators=150, learning_rate=0.05, max_depth=3, subsample=0.8, random_state=42
    ),
    'HistGradientBoosting (LightGBM-style)': HistGradientBoostingClassifier(
        max_iter=150, learning_rate=0.05, max_leaf_nodes=15, l2_regularization=1.0,
        early_stopping=True, random_state=42
    )
}

if HAS_XGB:
    modelos_gb['XGBoost Classifier'] = XGBClassifier(
        n_estimators=150, learning_rate=0.05, max_depth=3, subsample=0.8,
        colsample_bytree=0.8, reg_lambda=1.0, eval_metric='logloss', random_state=42
    )
if HAS_LGBM:
    modelos_gb['LightGBM Classifier'] = LGBMClassifier(
        n_estimators=150, learning_rate=0.05, num_leaves=15, subsample=0.8,
        colsample_bytree=0.8, reg_lambda=1.0, random_state=42, verbose=-1
    )
if HAS_CAT:
    modelos_gb['CatBoost Classifier'] = CatBoostClassifier(
        iterations=150, learning_rate=0.05, depth=4, l2_leaf_reg=3.0,
        verbose=0, random_state=42
    )

for nombre, modelo in modelos_gb.items():
    modelo.fit(X_train, y_train)
    preds = modelo.predict(X_test)
    probs = modelo.predict_proba(X_test)[:, 1]
    print(f"{nombre:<38} -> Acc: {accuracy_score(y_test, preds)*100:.2f}% | ROC-AUC: {roc_auc_score(y_test, probs):.4f}")


## 2. Manejo Nativo de Valores Faltantes (`NaN`)

Una ventaja decisiva de **XGBoost**, **LightGBM** y `HistGradientBoostingClassifier` sobre la Regresión Logística, SVM o KNN es su capacidad para **aprender automáticamente a qué rama (izquierda o derecha) enviar los valores `NaN`** durante la división de cada nodo, minimizando la pérdida sin requerir un `SimpleImputer` previo.


In [ ]:
# Simulacion de un 15% de valores faltantes (NaN) en el dataset
rng = np.random.RandomState(42)
X_train_nan = X_train.copy()
X_test_nan = X_test.copy()

mask_train = rng.rand(*X_train_nan.shape) < 0.15
mask_test = rng.rand(*X_test_nan.shape) < 0.15
X_train_nan[mask_train] = np.nan
X_test_nan[mask_test] = np.nan

hgb_nan = HistGradientBoostingClassifier(max_iter=120, learning_rate=0.05, random_state=42)
hgb_nan.fit(X_train_nan, y_train)

acc_nan = accuracy_score(y_test, hgb_nan.predict(X_test_nan))
auc_nan = roc_auc_score(y_test, hgb_nan.predict_proba(X_test_nan)[:, 1])
print(f"Rendimiento con 15% de valores NaN sin imputacion previa:")
print(f"-> Exactitud en Test: {acc_nan*100:.2f}% | ROC-AUC: {auc_nan:.4f}")


## 3. Comparación Visual de Fronteras de Decisión en 2D

Comparamos cómo dividen el espacio bidimensional un modelo de Bagging (`RandomForestClassifier`) frente a los esquemas de Boosting (`GradientBoosting` y `HistGradientBoosting`):


In [ ]:
X_2d, y_2d = make_moons(n_samples=350, noise=0.28, random_state=42)

clfs_2d = {
    'Random Forest (Bagging)': RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42),
    'Gradient Boosting (Depth=3)': GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42),
    'HistGradientBoosting (LightGBM)': HistGradientBoostingClassifier(max_iter=100, learning_rate=0.1, max_leaf_nodes=10, random_state=42)
}

fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))
for ax, (titulo, clf) in zip(axes, clfs_2d.items()):
    clf.fit(X_2d, y_2d)
    DecisionBoundaryDisplay.from_estimator(clf, X_2d, cmap='coolwarm', alpha=0.55, ax=ax, response_method='predict_proba')
    ax.scatter(X_2d[:, 0], X_2d[:, 1], c=y_2d, cmap='coolwarm', edgecolor='k', s=35)
    ax.set_title(f"{titulo}\n(Acc: {clf.score(X_2d, y_2d)*100:.1f}%)", fontweight='bold')
    ax.set_xlabel('Caracteristica 1')
    ax.set_ylabel('Caracteristica 2')

plt.suptitle('Fronteras de Probabilidad: Bagging vs Gradient Boosting', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## 4. Benchmark Integral de Clasificadores

Evaluamos el desempeño en exactitud, F1-Score, ROC-AUC y tiempos de entrenamiento/inferencia comparando los modelos clásicos del curso con los algoritmos de **Gradient Boosting**:


In [ ]:
suite_benchmark = {
    'Regresion Logistica': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=150, random_state=42, n_jobs=-1),
    'Gradient Boosting (Sklearn)': GradientBoostingClassifier(n_estimators=150, learning_rate=0.05, max_depth=3, random_state=42),
    'HistGradientBoosting (LGBM)': HistGradientBoostingClassifier(max_iter=150, learning_rate=0.05, random_state=42)
}
if HAS_XGB:
    suite_benchmark['XGBoost'] = modelos_gb['XGBoost Classifier']
if HAS_CAT:
    suite_benchmark['CatBoost'] = modelos_gb['CatBoost Classifier']

registros = []
for nombre, clf in suite_benchmark.items():
    t0 = time.perf_counter()
    clf.fit(X_train, y_train)
    t_train = (time.perf_counter() - t0) * 1000

    t0 = time.perf_counter()
    y_pred = clf.predict(X_test)
    y_prob = clf.predict_proba(X_test)[:, 1]
    t_pred = (time.perf_counter() - t0) * 1000

    registros.append({
        'Algoritmo': nombre,
        'Accuracy (%)': accuracy_score(y_test, y_pred) * 100,
        'F1-Score (%)': f1_score(y_test, y_pred) * 100,
        'ROC-AUC (%)': roc_auc_score(y_test, y_prob) * 100,
        'Train (ms)': t_train,
        'Inferencia (ms)': t_pred
    })

df_res = pd.DataFrame(registros).sort_values(by='ROC-AUC (%)', ascending=False)
print(df_res.to_string(index=False))


In [ ]:
# Importancia de Caracteristicas por Permutacion en el modelo Boosting
gbc_best = suite_benchmark['HistGradientBoosting (LGBM)']
perm_imp = permutation_importance(gbc_best, X_test, y_test, n_repeats=15, random_state=42, n_jobs=-1)

top_idx = np.argsort(perm_imp.importances_mean)[-10:]
plt.figure(figsize=(9, 4.8))
plt.barh(np.array(cancer.feature_names)[top_idx], perm_imp.importances_mean[top_idx], xerr=perm_imp.importances_std[top_idx], color='#2b5c8f')
plt.xlabel('Disminucion Media en Exactitud al Permutar', fontweight='bold')
plt.title('Top 10 Caracteristicas Mas Importantes (Permutation Importance)', fontweight='bold')
plt.tight_layout()
plt.show()
